# OpenSG on a Colab GPU — Drive-resident environment

**Runtime → Change runtime type → T4 GPU** first.

Two kinds of cell:

* **Cell 0 — BUILD THE ENV. Run once, ever.** Installs the packages
  Colab lacks into `OpenSG-2.0/_env` on your Drive.
* **Cell 1 — ACTIVATE. Run at the start of every session** (and
  after any interruption). ~15 s: mounts Drive, puts the Drive env
  and `src/` on the path, turns on float64, points the JAX
  compilation cache at Drive.

Nothing is ever re-downloaded and nothing is cloned; OpenSG is pure
Python, so `src/` on Drive is used in place.

In [ ]:
#@title 0) BUILD THE ENV — run ONCE (skip it on later sessions)
ROOT = '/content/drive/MyDrive/OpenSG-2.0'  #@param {type:"string"}
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
ENV = os.path.join(ROOT, '_env')
os.makedirs(ENV, exist_ok=True)

# Colab already ships jax(+CUDA), scipy, pyyaml, matplotlib -- only
# install what is genuinely missing, so the Drive env stays small and
# loads fast.  pypardiso = MKL PARDISO, the direct solver's fast
# backend; without it `--solver direct` falls back to SciPy SuperLU
# (20-200x slower) and every CPU-vs-GPU number is unfair.
need = []
for mod, pkg in (('pypardiso', 'pypardiso'), ('yaml', 'pyyaml'),
                 ('scipy', 'scipy'), ('matplotlib', 'matplotlib')):
    try:
        __import__(mod)
    except ImportError:
        need.append(pkg)
print('installing into the Drive env:', need or '(nothing missing)')
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--target', ENV] + need, check=True)

import jax
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('\njax sees no GPU. If the runtime IS a GPU one, install the'
          ' CUDA plugin (into the SESSION, not Drive -- it is ~2 GB):')
    print('    !pip -q install --upgrade "jax[cuda12]"')
    print('then Runtime > Restart session and run cell 1.')
print('\nenv ready:', ENV)
!du -sh {ENV}

In [ ]:
#@title 1) ACTIVATE — run this at the start of EVERY session
ROOT = '/content/drive/MyDrive/OpenSG-2.0'  #@param {type:"string"}
import os, sys, glob
from google.colab import drive
drive.mount('/content/drive')

SRC, ENV = os.path.join(ROOT, 'src'), os.path.join(ROOT, '_env')
if not os.path.isdir(SRC):
    print('NOT FOUND:', SRC, '\nDrive folders that look close:')
    for p in (glob.glob('/content/drive/MyDrive/*OpenSG*')
              + glob.glob('/content/drive/MyDrive/*/*OpenSG*')):
        print('   ', p)
    raise SystemExit('set ROOT to the folder that CONTAINS src/')

for p in (ENV, SRC):                       # src first on the path
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
os.environ['PYTHONPATH'] = SRC + (os.pathsep + ENV
                                  if os.path.isdir(ENV) else '')
os.environ['JAX_ENABLE_X64'] = '1'

CACHE = os.path.join(ROOT, '_jax_cache')   # JIT work survives a kill
os.makedirs(CACHE, exist_ok=True)
import jax, jax.numpy as jnp
jax.config.update('jax_enable_x64', True)
jax.config.update('jax_compilation_cache_dir', CACHE)
jax.config.update('jax_persistent_cache_min_entry_size_bytes', -1)
jax.config.update('jax_persistent_cache_min_compile_time_secs', 1.0)

print('src     :', SRC)
print('env     :', ENV if os.path.isdir(ENV) else '(none -- run cell 0)')
print('devices :', jax.devices())
print('dtype   :', jnp.zeros(3).dtype, ' (MUST be float64)')
assert jnp.zeros(3).dtype == jnp.float64, 'x64 is off -- stop here'
try:
    import pypardiso; print('pypardiso: yes (fast direct solver)')
except ImportError:
    print('pypardiso: NO -- direct falls back to SuperLU, 20-200x slower')
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('\nNO GPU VISIBLE -- Runtime > Change runtime type > T4 GPU')

In [ ]:
#@title 2) Smoke test: package + material library resolve from Drive
import opensg_solid
from opensg_solid.io.materials_db import load_library
lib = load_library()
print('opensg_solid :', opensg_solid.__file__)
print('library      :', lib['path'])
print('materials    :', sorted(lib['materials'])[:8], '...')
!python -m opensg --help | head -5

In [ ]:
#@title 3) Mesh -> yaml ON COLAB (upload only the small LINEAR .msh)
#@markdown `--p_refine` builds the tet10 twin here, so neither the
#@markdown quadratic mesh nor its multi-GB yaml crosses the network.
MESH = '/content/drive/MyDrive/meshes/SP_solid_rho0.3_n5.09714.msh'  #@param {type:"string"}
WORK = '/content/work'  #@param {type:"string"}
import os, shutil
os.makedirs(WORK, exist_ok=True)
local = os.path.join(WORK, os.path.basename(MESH))
if not os.path.exists(local):
    shutil.copy(MESH, local)     # Drive is FUSE: solve on local disk
!cd {WORK} && python -m opensg msh_to_yaml {os.path.basename(local)} --mat1 Al --n_model 2 --refined 1 --p_refine
!ls -la {WORK}/*.yaml

In [ ]:
#@title 4) BENCHMARK -- GPU (cg) vs CPU-direct on the same law
DRIVER = os.path.join(ROOT, 'gpu_vs_direct.py')
!cd {WORK} && python {DRIVER} *.yaml --solvers direct,cg --repeat 1 --out {WORK}/solver_benchmark.dat
print(open(f'{WORK}/solver_benchmark.dat').read())

In [ ]:
#@title 5) Big case, GPU only (skip direct past ~1M dofs)
BIG = 'SP_solid_rho0.3_n5.09714_quad.yaml'  #@param {type:"string"}
!cd {WORK} && python {DRIVER} {BIG} --solvers cg --repeat 2 --out {WORK}/big_gpu.dat
print(open(f'{WORK}/big_gpu.dat').read())

In [ ]:
#@title 6) Save results to Drive (survives the session)
import glob, shutil
OUT = os.path.join(ROOT, 'colab_results')
os.makedirs(OUT, exist_ok=True)
for f in glob.glob(f'{WORK}/*.dat') + glob.glob(f'{WORK}/*.out'):
    shutil.copy(f, OUT); print('saved', os.path.basename(f))
print('->', OUT)

## Notes

* **Session died?** Run cell 1 only. Source, env and JAX cache are on
  Drive; only work that was mid-solve is lost.
* **Why `--target` and not a venv?** The Colab kernel is fixed to its
  own interpreter, so a venv would not apply to code cells. A
  `pip --target` directory on Drive plus `sys.path` gives the same
  persistence and works for both code cells and `!python` cells.
* **Do not put a CUDA jax on Drive.** It is ~2 GB of shared objects
  and loads slowly over FUSE; Colab's GPU runtime already provides
  one. Install it into the session if cell 1 reports no GPU.
* **Keep working files on `/content`**, not Drive — FUSE is fine for
  reading a mesh once, painful for solver intermediates.
* **`rel diff` must be ~1e-8 or smaller** between `cg` and `direct`;
  the CG error enters the stiffness at *second* order of the solve
  tolerance. Near 1e-3 means float32 slipped in — cell 1 asserts.
* **T4 = 16 GB**: a ~2M-dof tet10 SG fits matrix-free; ~20M dofs is
  the host-resident `--solver stream` (iter 3) route, not a T4.